# DeepSets Hyperparameter Tuning

This notebook implements hyperparameter search for the DeepSets quantum error correction model.

**Purpose:** Find optimal DeepSets architecture for comparison with GraphSAGE.

**Key Advantage:** DeepSets handles variable-size inputs natively, enabling training on mixed distances
and extrapolation to larger distances without truncation.

**Workflow:**
1. Generate flat dataset cache (if not exists)
2. Run the training loop (trains all 50 configs sequentially)
3. Run the analysis cells to find the best configuration

**Distributed Mode (optional):**
Set `DISTRIBUTED_MODE = True` and `WORKER_ID` to distribute across multiple machines.

In [8]:
#==============================================================================
# MODE CONFIGURATION
#==============================================================================
DISTRIBUTED_MODE = False  # Set to True for multi-machine distributed training
WORKER_ID = 1  # Only used if DISTRIBUTED_MODE = True (set to 1-5)
#==============================================================================

if DISTRIBUTED_MODE:
    assert WORKER_ID in [1, 2, 3, 4, 5], f"WORKER_ID must be 1, 2, 3, 4, or 5, got {WORKER_ID}"
    if WORKER_ID == 5:
        my_config_ids = list(range(40, 50))
    else:
        my_config_ids = [i for i in range(40) if i % 4 == (WORKER_ID - 1)]
    print(f"DISTRIBUTED MODE - Worker {WORKER_ID}")
    print(f"This worker will train {len(my_config_ids)} configs: {my_config_ids}")
else:
    my_config_ids = list(range(50))  # Train all 50 configs
    print(f"LOCAL MODE - Training all 50 configs sequentially")

LOCAL MODE - Training all 50 configs sequentially


## Imports

In [9]:
# Install required libraries (uncomment and run if needed)
# !pip install stim pymatching numpy matplotlib torch tqdm

In [10]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/deepsets/tuning -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Import from benchmark_models.py
from benchmark_models import (
    SurfaceCodeSampler,
    DeepSetsModel,
    DeepSets,
    FlatDatasetCache,
)

# Set up paths
TUNING_DIR = BASE_PATH / "deepsets" / "tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = TUNING_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = TUNING_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = TUNING_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  TUNING_DIR: {TUNING_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

Using device: cuda
GPU: NVIDIA GeForce GTX 1080

Paths:
  BASE_PATH: ..\..
  TUNING_DIR: ..\..\deepsets\tuning
  RESULTS_DIR: ..\..\deepsets\tuning\results
  MODELS_DIR: ..\..\deepsets\tuning\models


## Generate Configs

Config generation - 50 configurations for hyperparameter search.
The `configs.json` file will be created on first run.

In [11]:
# =============================================================================
# CONFIG GENERATION - RUN ONCE
# =============================================================================

CONFIGS_PATH = TUNING_DIR / "configs.json"

# Check if configs already exist
if CONFIGS_PATH.exists():
    print(f"configs.json already exists at {CONFIGS_PATH}")
    print("Skipping generation. Delete the file to regenerate.")
else:
    # Search space definition for DeepSets
    SEARCH_SPACE = {
        'phi_hidden': [(64,), (128,), (128, 128), (256, 128)],
        'rho_hidden': [(128,), (256,), (256, 128), (512, 256)],
        'pool': ['mean', 'sum'],
        'learning_rate': [1e-4, 3e-4, 1e-3],
        'batch_size': [32, 64, 128],
        'dropout': [0.0, 0.1, 0.2],
    }

    # Fixed parameters (not searched)
    FIXED_PARAMS = {
        'epochs': 30,
        'distance': 7,
    }

    # Number of configurations to generate
    N_CONFIGS = 50
    SEED = 42

    # Set seed for reproducibility
    random.seed(SEED)

    # Generate random configurations
    configs = []
    for i in range(N_CONFIGS):
        config = {
            'id': i,
            'phi_hidden': random.choice(SEARCH_SPACE['phi_hidden']),
            'rho_hidden': random.choice(SEARCH_SPACE['rho_hidden']),
            'pool': random.choice(SEARCH_SPACE['pool']),
            'learning_rate': random.choice(SEARCH_SPACE['learning_rate']),
            'batch_size': random.choice(SEARCH_SPACE['batch_size']),
            'dropout': random.choice(SEARCH_SPACE['dropout']),
        }
        configs.append(config)

    # Create the full config document
    config_doc = {
        'seed': SEED,
        'n_configs': N_CONFIGS,
        'generated_at': datetime.now().isoformat(),
        'search_space': {k: [str(v) for v in vals] for k, vals in SEARCH_SPACE.items()},
        'fixed_params': FIXED_PARAMS,
        'configs': configs
    }

    # Save to file
    with open(CONFIGS_PATH, 'w') as f:
        json.dump(config_doc, f, indent=2)

    print(f"Generated {N_CONFIGS} configurations with seed {SEED}")
    print(f"Saved to: {CONFIGS_PATH}")
    print(f"\nSearch space:")
    for key, values in SEARCH_SPACE.items():
        print(f"  {key}: {values}")
    print(f"\nFixed parameters:")
    for key, value in FIXED_PARAMS.items():
        print(f"  {key}: {value}")
    print(f"\nFirst 5 configs:")
    for c in configs[:5]:
        print(f"  {c}")

configs.json already exists at ..\..\deepsets\tuning\configs.json
Skipping generation. Delete the file to regenerate.


## Training Loop

This is the main training cell. It will:
1. Load the configs assigned to this worker
2. Skip any configs already completed (for resume support)
3. Train each config and save results incrementally

In [12]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_configs():
    """Load all configurations from configs.json."""
    configs_path = TUNING_DIR / "configs.json"
    if not configs_path.exists():
        raise FileNotFoundError(f"configs.json not found at {configs_path}. Run the config generation cell first.")
    with open(configs_path, 'r') as f:
        return json.load(f)

def get_my_configs(all_configs, config_ids):
    """Get configs by ID list."""
    return [c for c in all_configs['configs'] if c['id'] in config_ids]

def get_completed_ids():
    """Get set of already-completed config IDs from results file."""
    results_path = RESULTS_DIR / "results.json"
    if not results_path.exists():
        return set()
    with open(results_path, 'r') as f:
        results = json.load(f)
    return {r['config_id'] for r in results if r.get('status') == 'completed'}

def save_result(result):
    """Append a result to the results file."""
    results_path = RESULTS_DIR / "results.json"

    if results_path.exists():
        with open(results_path, 'r') as f:
            results = json.load(f)
    else:
        results = []

    results.append(result)

    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    return len(results)

def evaluate_deepsets(model, detections, labels, device, batch_size=256):
    """Evaluate DeepSets model accuracy on a set of samples."""
    model.model.eval()

    correct = 0
    total = len(labels)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            X = detections[i:i+batch_size].unsqueeze(-1)  # [batch, N, 1]
            y = labels[i:i+batch_size]
            pred = model.model(X)
            correct += ((pred.squeeze() > 0.5).float() == y).sum().item()

    return correct / total if total > 0 else 0.0

def load_and_split_dataset(cache_name, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Load flat dataset from cache and split into train/val/test."""
    print(f"Loading flat dataset: {cache_name}")
    cache = FlatDatasetCache(base_path=BASE_PATH, device=device)
    cache.load(cache_name, verbose=True)

    detections, labels = cache.get_data()
    n = len(labels)

    # Shuffle with fixed seed
    torch.manual_seed(seed)
    perm = torch.randperm(n, device=device)
    detections = detections[perm]
    labels = labels[perm]

    # Calculate split points
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_det = detections[:train_end]
    train_lab = labels[:train_end]
    val_det = detections[train_end:val_end]
    val_lab = labels[train_end:val_end]
    test_det = detections[val_end:]
    test_lab = labels[val_end:]

    print(f"Dataset split: {len(train_lab)} train, {len(val_lab)} val, {len(test_lab)} test")

    return (train_det, train_lab), (val_det, val_lab), (test_det, test_lab)

In [13]:
# =============================================================================
# LOAD CONFIGS AND DATASET
# =============================================================================

import gc

# Load all configurations
all_configs = load_configs()
fixed_params = all_configs['fixed_params']

# Get configs to train (based on mode setting from above)
my_configs = get_my_configs(all_configs, my_config_ids)
print(f"\nConfigs to train: {len(my_configs)}")
print(f"Config IDs: {[c['id'] for c in my_configs]}")

# Check which are already completed
completed_ids = get_completed_ids()
remaining_configs = [c for c in my_configs if c['id'] not in completed_ids]
print(f"\nAlready completed: {len(completed_ids)} configs")
print(f"Remaining: {len(remaining_configs)} configs")

if len(remaining_configs) == 0:
    print("\nAll configs are already completed!")
else:
    # Load dataset once (shared across all configs)
    cache_name = f"d{fixed_params['distance']}_baseline"
    (train_det, train_lab), (val_det, val_lab), (test_det, test_lab) = load_and_split_dataset(cache_name)


Configs to train: 50
Config IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]

Already completed: 0 configs
Remaining: 50 configs
Loading flat dataset: d7_baseline
Loading flat dataset 'd7_baseline' (1289.4 MB)...
  Loaded 1,000,000 samples
  Detections shape: torch.Size([1000000, 336])
  Distance: d=7
  Error rates: [0.001, 0.003, 0.005, 0.007]
Dataset split: 800000 train, 100000 val, 100000 test


In [14]:
if len(remaining_configs) > 0:
    print(f"\n{'='*60}")
    print(f"Starting training - {len(remaining_configs)} configs")
    print(f"{'='*60}")

    for i, config in enumerate(remaining_configs):
        config_id = config['id']
        print(f"\n{'='*60}")
        print(f"Config {config_id} ({i+1}/{len(remaining_configs)})")
        print(f"{'='*60}")
        print(f"Parameters:")
        print(f"  phi_hidden: {config['phi_hidden']}")
        print(f"  rho_hidden: {config['rho_hidden']}")
        print(f"  pool: {config['pool']}")
        print(f"  learning_rate: {config['learning_rate']}")
        print(f"  batch_size: {config['batch_size']}")
        print(f"  dropout: {config['dropout']}")

        start_time = time.time()

        try:
            # Initialize model with config hyperparameters
            phi_hidden = tuple(config['phi_hidden']) if isinstance(config['phi_hidden'], list) else config['phi_hidden']
            rho_hidden = tuple(config['rho_hidden']) if isinstance(config['rho_hidden'], list) else config['rho_hidden']

            model = DeepSets(
                nickname=f"config_{config_id}",
                phi_hidden=phi_hidden,
                rho_hidden=rho_hidden,
                pool=config['pool'],
                dropout=config['dropout'],
                device=device,
                base_path=BASE_PATH,
                seed=config_id  # Use config_id as seed for reproducibility
            )

            # Train the model
            epoch_losses = model.train_from_data(
                detections=train_det,
                labels=train_lab,
                epochs=fixed_params['epochs'],
                batch_size=config['batch_size'],
                lr=config['learning_rate'],
                verbose=True
            )

            # Evaluate on validation and test sets
            val_accuracy = evaluate_deepsets(model, val_det, val_lab, device)
            test_accuracy = evaluate_deepsets(model, test_det, test_lab, device)

            training_time = time.time() - start_time

            # Save model checkpoint
            model_path = MODELS_DIR / f"config_{config_id}.pt"
            torch.save({
                'state_dict': model.model.state_dict(),
                'config': config,
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
            }, model_path)

            # Record result
            result = {
                'config_id': config_id,
                'config': config,
                'status': 'completed',
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'final_loss': epoch_losses[-1] if epoch_losses else None,
                'training_time_seconds': training_time,
                'timestamp': datetime.now().isoformat(),
            }

            n_saved = save_result(result)

            print(f"\nConfig {config_id} COMPLETED")
            print(f"  Val accuracy: {val_accuracy:.4f}")
            print(f"  Test accuracy: {test_accuracy:.4f}")
            print(f"  Final loss: {epoch_losses[-1]:.4f}")
            print(f"  Training time: {training_time:.1f}s")
            print(f"  Total results saved: {n_saved}")

        except Exception as e:
            print(f"\nConfig {config_id} FAILED: {e}")
            result = {
                'config_id': config_id,
                'config': config,
                'status': 'failed',
                'error': str(e),
                'timestamp': datetime.now().isoformat(),
            }
            save_result(result)

        finally:
            # Clean up GPU memory
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n{'='*60}")
    print("ALL TRAINING COMPLETE")
    print(f"{'='*60}")


Starting training - 50 configs

Config 0 (1/50)
Parameters:
  phi_hidden: [64]
  rho_hidden: [128]
  pool: sum
  learning_rate: 0.0001
  batch_size: 32
  dropout: 0.0
DeepSets initialized: DeepSets(nickname='config_0', phi_hidden=(64,), rho_hidden=(128,), pool='sum', dropout=0.0)

Training: DeepSets(nickname='config_0', phi_hidden=(64,), rho_hidden=(128,), pool='sum', dropout=0.0)
Epochs: 10 | Batch size: 32 | LR: 0.0001
Training samples: 800000
Input shape: torch.Size([800000, 336])


Epoch 1/10: 100%|██████████| 25000/25000 [01:17<00:00, 321.52it/s, loss=68.7500, acc=0.2992]


Epoch 1/10 - Loss: 72.2498, Acc: 0.2958


Epoch 2/10: 100%|██████████| 25000/25000 [01:23<00:00, 300.94it/s, loss=62.5000, acc=0.3063]


Epoch 2/10 - Loss: 72.2498, Acc: 0.2865


...

Epoch 1/10 - Loss: 0.5516, Acc: 0.7245


Epoch 2/10: 100%|██████████| 12500/12500 [00:57<00:00, 215.76it/s, loss=0.5214, acc=0.7252]


Epoch 2/10 - Loss: 0.5496, Acc: 0.7286


Epoch 3/10:  48%|████▊     | 5993/12500 [00:28<00:31, 207.93it/s, loss=0.5278, acc=0.7327]


KeyboardInterrupt: 

## Results Analysis

Load and analyze the tuning results to find the best configuration.

In [ ]:
# =============================================================================
# LOAD AND ANALYZE RESULTS
# =============================================================================

results_path = RESULTS_DIR / "results.json"

if results_path.exists():
    with open(results_path, 'r') as f:
        results = json.load(f)

    # Filter to completed results only
    completed = [r for r in results if r.get('status') == 'completed']
    print(f"Completed configs: {len(completed)} / {len(results)}")

    if completed:
        # Create DataFrame for analysis
        df_data = []
        for r in completed:
            config = r['config']
            df_data.append({
                'config_id': r['config_id'],
                'phi_hidden': str(config['phi_hidden']),
                'rho_hidden': str(config['rho_hidden']),
                'pool': config['pool'],
                'learning_rate': config['learning_rate'],
                'batch_size': config['batch_size'],
                'dropout': config['dropout'],
                'val_accuracy': r['val_accuracy'],
                'test_accuracy': r['test_accuracy'],
                'final_loss': r.get('final_loss'),
                'training_time': r.get('training_time_seconds', 0),
            })

        df = pd.DataFrame(df_data)
        df_sorted = df.sort_values('test_accuracy', ascending=False)

        print("\nTop 10 Configurations by Test Accuracy:")
        print(df_sorted.head(10).to_string(index=False))

        # Best config
        best = df_sorted.iloc[0]
        print(f"\n{'='*60}")
        print("BEST CONFIGURATION")
        print(f"{'='*60}")
        print(f"Config ID: {int(best['config_id'])}")
        print(f"Test Accuracy: {best['test_accuracy']:.4f}")
        print(f"Val Accuracy: {best['val_accuracy']:.4f}")
        print(f"\nHyperparameters:")
        print(f"  phi_hidden: {best['phi_hidden']}")
        print(f"  rho_hidden: {best['rho_hidden']}")
        print(f"  pool: {best['pool']}")
        print(f"  learning_rate: {best['learning_rate']}")
        print(f"  batch_size: {int(best['batch_size'])}")
        print(f"  dropout: {best['dropout']}")
else:
    print("No results file found. Run the training loop first.")

In [ ]:
# =============================================================================
# SAVE BEST CONFIG
# =============================================================================

if results_path.exists() and len(completed) > 0:
    best_config = completed[df_sorted.index[0]]['config']
    best_result = {
        'config_id': int(best['config_id']),
        'phi_hidden': best_config['phi_hidden'],
        'rho_hidden': best_config['rho_hidden'],
        'pool': best_config['pool'],
        'learning_rate': best_config['learning_rate'],
        'batch_size': best_config['batch_size'],
        'dropout': best_config['dropout'],
        'test_accuracy': best['test_accuracy'],
        'val_accuracy': best['val_accuracy'],
    }

    best_config_path = TUNING_DIR / "best_model_config.json"
    with open(best_config_path, 'w') as f:
        json.dump(best_result, f, indent=2)

    print(f"Best config saved to: {best_config_path}")

In [ ]:
# =============================================================================
# VISUALIZE RESULTS
# =============================================================================

if results_path.exists() and len(completed) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Test accuracy distribution
    ax1 = axes[0, 0]
    ax1.hist(df['test_accuracy'], bins=20, edgecolor='black', alpha=0.7)
    ax1.axvline(best['test_accuracy'], color='red', linestyle='--', label=f"Best: {best['test_accuracy']:.4f}")
    ax1.set_xlabel('Test Accuracy')
    ax1.set_ylabel('Count')
    ax1.set_title('Test Accuracy Distribution')
    ax1.legend()

    # 2. Accuracy by pool type
    ax2 = axes[0, 1]
    pool_groups = df.groupby('pool')['test_accuracy'].mean()
    pool_groups.plot(kind='bar', ax=ax2, color=['steelblue', 'coral'])
    ax2.set_xlabel('Pool Type')
    ax2.set_ylabel('Mean Test Accuracy')
    ax2.set_title('Accuracy by Pooling Method')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

    # 3. Accuracy by learning rate
    ax3 = axes[1, 0]
    lr_groups = df.groupby('learning_rate')['test_accuracy'].mean()
    lr_groups.plot(kind='bar', ax=ax3, color='steelblue')
    ax3.set_xlabel('Learning Rate')
    ax3.set_ylabel('Mean Test Accuracy')
    ax3.set_title('Accuracy by Learning Rate')
    ax3.set_xticklabels([f"{x:.0e}" for x in lr_groups.index], rotation=45)

    # 4. Accuracy by phi_hidden
    ax4 = axes[1, 1]
    phi_groups = df.groupby('phi_hidden')['test_accuracy'].mean().sort_values(ascending=False)
    phi_groups.plot(kind='bar', ax=ax4, color='steelblue')
    ax4.set_xlabel('Phi Hidden Dims')
    ax4.set_ylabel('Mean Test Accuracy')
    ax4.set_title('Accuracy by Phi Network Size')
    ax4.set_xticklabels(ax4.get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()

    # Save plot
    plot_path = PLOTS_DIR / "tuning_results.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Plot saved to: {plot_path}")

    plt.show()